# Voix de Kodo : accent africain

Ce carnet fabrique une voix de synthèse française **à accent africain** pour Kodo, gratuitement, sur une carte graphique prêtée par Google Colab.

**Avant de commencer :** menu *Exécution → Modifier le type d'exécution → GPU T4*. Puis lance les cellules **une par une, dans l'ordre** (bouton ▶ à gauche de chaque cellule).

Ce qu'il fait :
1. Il installe Piper, le moteur de voix libre utilisé par Douke.
2. Il télécharge le corpus libre **African Accented French** (OpenSLR 57, licence Apache 2.0, 1,8 Go) : des enregistrements de personnes du Cameroun, du Tchad, du Congo, du Gabon et du Niger.
3. Il te fait **écouter les voix** qui ont le plus d'enregistrements, pour que tu choisisses celle de Kodo.
4. Il adapte la voix française « siwis » de Piper à la voix choisie (environ 2 à 3 heures).
5. Il te fait écouter le résultat, puis prépare les deux fichiers à déposer sur GitHub.

Tout se passe sur les machines de Google : rien n'est téléchargé sur ton ordinateur, sauf les deux fichiers finaux (environ 60 Mo).

## 1. Brancher Google Drive

L'entraînement est long et Colab peut se couper : les sauvegardes vont donc dans ton Google Drive, pour reprendre là où on s'est arrêté. Autorise l'accès quand Google le demande.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
TRAVAIL = '/content/drive/MyDrive/voix-kodo'
os.makedirs(TRAVAIL, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Installer Piper (environ 5 minutes)

In [ ]:
%%bash
set -e
apt-get -qq update && apt-get -qq install -y build-essential cmake ninja-build espeak-ng > /dev/null
cd /content
[ -d piper1-gpl ] || git clone -q https://github.com/OHF-voice/piper1-gpl.git
cd piper1-gpl
pip -q install -e '.[train]'
./build_monotonic_align.sh > /dev/null
python3 setup.py build_ext --inplace > /dev/null
echo "Piper est prêt."

## 3. Télécharger le corpus African Accented French (environ 10 minutes)

1,8 Go, depuis OpenSLR (licence Apache 2.0). Il est gardé dans ton Drive : si tu relances le carnet, il n'est pas retéléchargé.

In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/voix-kodo
if [ ! -f African_Accented_French.tar.gz ]; then
  wget -q --show-progress https://www.openslr.org/resources/57/African_Accented_French.tar.gz \
    || wget -q --show-progress https://openslr.elda.org/resources/57/African_Accented_French.tar.gz
fi
mkdir -p /content/corpus
tar -xzf African_Accented_French.tar.gz -C /content/corpus
echo "Corpus extrait :"; find /content/corpus -maxdepth 3 -type d | head -30

## 4. Trouver les voix qui ont le plus d'enregistrements

On associe chaque fichier audio à sa transcription, puis on range les personnes par durée totale. La cellule cherche les transcriptions dans tous les fichiers texte du corpus et garde les enregistrements qu'elle sait relier à un texte.

In [ ]:
import glob, os, re, collections, soundfile as sf

wavs = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob('/content/corpus/**/*.wav', recursive=True)}
print(len(wavs), 'fichiers audio')

textes = {}
for fichier in glob.glob('/content/corpus/**/*', recursive=True):
    if not os.path.isfile(fichier) or fichier.endswith(('.wav', '.tar.gz')):
        continue
    try:
        lignes = open(fichier, encoding='utf-8', errors='ignore').read().splitlines()
    except Exception:
        continue
    for ligne in lignes:
        morceaux = re.split(r'\t|\s+', ligne.strip(), maxsplit=1)
        if len(morceaux) == 2:
            cle = os.path.splitext(os.path.basename(morceaux[0]))[0]
            if cle in wavs and len(morceaux[1]) > 3:
                textes[cle] = morceaux[1].strip()
print(len(textes), 'fichiers reliés à une transcription')

# Le locuteur : le dossier qui contient le fichier audio.
duree = collections.Counter()
par_locuteur = collections.defaultdict(list)
for cle, texte in textes.items():
    chemin = wavs[cle]
    info = sf.info(chemin)
    d = info.frames / info.samplerate
    if 1.0 <= d <= 15.0:
        locuteur = os.path.relpath(os.path.dirname(chemin), '/content/corpus')
        duree[locuteur] += d
        par_locuteur[locuteur].append((cle, texte, d))

print('\nLes 15 voix les plus fournies :')
for i, (loc, d) in enumerate(duree.most_common(15)):
    print(f'{i:2d}. {loc:60s} {d/60:5.1f} min, {len(par_locuteur[loc])} phrases')

## 5. Écouter et choisir

Lance la cellule : elle joue trois phrases de chacune des 8 voix les plus fournies. Note les numéros des voix qui te plaisent et dont l'accent convient à Kodo.

**Conseil :** une seule voix d'au moins 20 minutes donne la voix la plus nette. Si aucune n'atteint cette durée, choisis plusieurs voix **du même genre et très proches** : elles seront fondues en une seule.

In [ ]:
from IPython.display import Audio, display, HTML
for i, (loc, d) in enumerate(duree.most_common(8)):
    display(HTML(f'<h4>Voix {i} : {loc} ({d/60:.1f} min)</h4>'))
    for cle, texte, _ in par_locuteur[loc][:3]:
        display(HTML(f'<i>{texte}</i>'))
        display(Audio(wavs[cle]))

## 6. Préparer les enregistrements de la voix choisie

Remplace les numéros dans `CHOIX` par ceux des voix retenues (par exemple `[0]` ou `[1, 4]`), puis lance la cellule. Les fichiers sont convertis au format de Piper (22 050 Hz, mono, volume égalisé, silences coupés).

In [ ]:
CHOIX = [0]

import numpy as np, librosa, csv
classement = [loc for loc, _ in duree.most_common()]
choisis = [classement[i] for i in CHOIX]
DONNEES = '/content/donnees'
os.makedirs(f'{DONNEES}/audio', exist_ok=True)
lignes, total = [], 0.0
for loc in choisis:
    for cle, texte, d in par_locuteur[loc]:
        audio, _ = librosa.load(wavs[cle], sr=22050, mono=True)
        audio, _ = librosa.effects.trim(audio, top_db=35)
        pic = np.max(np.abs(audio)) or 1.0
        sf.write(f'{DONNEES}/audio/{cle}.wav', (audio / pic * 0.9).astype(np.float32), 22050, subtype='PCM_16')
        lignes.append((f'{cle}.wav', texte))
        total += len(audio) / 22050
with open(f'{DONNEES}/metadata.csv', 'w', encoding='utf-8', newline='') as f:
    csv.writer(f, delimiter='|').writerows(lignes)
print(f'{len(lignes)} phrases, {total/60:.1f} minutes de voix pour Kodo.')

## 7. Entraîner la voix (environ 2 à 3 heures)

On part de la voix française « siwis » de Piper (licence CC-BY 4.0), qu'on adapte à la voix choisie : c'est beaucoup plus rapide que de partir de zéro.

Les sauvegardes vont dans ton Drive (`voix-kodo/entrainement`). **Si Colab se coupe**, relance les cellules 1, 2, 4 et 6, puis celle-ci : l'entraînement reprend au dernier point de sauvegarde.

Laisse l'onglet ouvert. Après une heure environ, tu peux arrêter la cellule (bouton ■) quand tu veux : la dernière sauvegarde sera utilisée.

In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/voix-kodo
DEPART=siwis.ckpt
if [ ! -f "$DEPART" ]; then
  wget -q --show-progress -O "$DEPART" "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/fr/fr_FR/siwis/medium/epoch%3D3304-step%3D2050940.ckpt"
fi
# Reprise : la dernière sauvegarde de cet entraînement, sinon la voix siwis.
REPRISE=$(ls -t entrainement/lightning_logs/*/checkpoints/*.ckpt 2>/dev/null | head -1 || true)
CKPT=${REPRISE:-$DEPART}
echo "Départ : $CKPT"
cd /content/piper1-gpl
python3 -m piper.train fit \
  --data.voice_name "kodo_afrique" \
  --data.csv_path /content/donnees/metadata.csv \
  --data.audio_dir /content/donnees/audio \
  --model.sample_rate 22050 \
  --data.espeak_voice "fr" \
  --data.cache_dir /content/cache \
  --data.config_path /content/drive/MyDrive/voix-kodo/afrique.onnx.json \
  --data.batch_size 16 \
  --trainer.max_epochs 6000 \
  --trainer.default_root_dir /content/drive/MyDrive/voix-kodo/entrainement \
  --ckpt_path "/content/drive/MyDrive/voix-kodo/$CKPT"

## 8. Exporter la voix et l'écouter

In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/voix-kodo
DERNIER=$(ls -t entrainement/lightning_logs/*/checkpoints/*.ckpt | head -1)
echo "Sauvegarde utilisée : $DERNIER"
cd /content/piper1-gpl
python3 -m piper.train.export_onnx --checkpoint "/content/drive/MyDrive/voix-kodo/$DERNIER" --output-file /content/drive/MyDrive/voix-kodo/afrique.onnx
ls -la /content/drive/MyDrive/voix-kodo/afrique.onnx*

In [ ]:
import wave
from IPython.display import Audio, display, HTML
from piper import PiperVoice
voix = PiperVoice.load('/content/drive/MyDrive/voix-kodo/afrique.onnx')
phrases = [
    'Salut, moi c’est Kodo ! On écrit ta première page web ?',
    'Bravo ! La balise h1 crée le grand titre de la page.',
    'Pas grave, chaque erreur t’apprend quelque chose. On réessaie ?',
]
for i, p in enumerate(phrases):
    with wave.open(f'/content/essai{i}.wav', 'wb') as w:
        voix.synthesize_wav(p, w)
    display(HTML(f'<i>{p}</i>'))
    display(Audio(f'/content/essai{i}.wav'))

## 9. Donner la voix à Douke

Si le résultat te plaît :

1. Dans ton Google Drive, dossier **voix-kodo**, télécharge les deux fichiers **afrique.onnx** et **afrique.onnx.json**.
2. Sur GitHub, ouvre le dépôt **douke-releases** → *Releases* → *Draft a new release*.
3. Dans *Choose a tag*, écris **voix-afrique** et crée-le. Titre : *Voix de Kodo, accent africain*.
4. Glisse les deux fichiers dans la zone des pièces jointes, puis publie la release.
5. Préviens-moi : je génère les fichiers audio avec cette voix et je la mets par défaut dans l'app.

Si le résultat ne te plaît pas, reviens à l'étape 5 et essaie d'autres voix, ou laisse l'entraînement tourner plus longtemps (étape 7).

*Crédits à afficher dans l'app : corpus African Accented French (OpenSLR 57, Apache 2.0) ; voix de départ « siwis » (données SIWIS, CC-BY 4.0) ; moteur Piper.*